In [1]:
using HomotopyContinuation
using LinearAlgebra

In [122]:
timesteps=4
@var m g R[1:timesteps,1:3,1:3] Cd[1:3,1:3] u[1:timesteps,1:4] v[1:timesteps,1:3] a_o[1:timesteps,1:3] w[1:timesteps,1:3] Σ⁻[1:3,1:3]
Bf=[0 0 0 0;0 0 0 0;1 1 1 1];
gravity=[0;0;-g];
relative_velocity=v-w;
airspeed=sqrt.(sum(relative_velocity.*relative_velocity,dims=2));
wind_changes=w[2:timesteps,:]-w[1:timesteps-1,:];
a_predicted=stack([gravity+(R[i,:,:]*Bf*u[i,:]-R[i,:,:]*Cd*transpose(R[i,:,:])*airspeed[i]*relative_velocity[i,:])/m for i in 1:timesteps],dims=1);

In [123]:
error=a_o-a_predicted
wind_change_losses=sum(wind_changes.*(wind_changes*transpose(Σ⁻)),dims=2)#1 fewer than there are timesteps
loss=sum(error.*error,dims=2)
total_loss=sum(loss)+sum(wind_change_losses);

In [227]:
constant_term=stack([a_o[i,:]-gravity-R[i,:,:]*Bf*u[i,:]/m for i in 1:timesteps],dims=1)
@var s[1:timesteps]#airspeed
I3=[1 0 0;0 1 0;0 0 1];
s_times_accel_loss_deriv=stack([transpose(transpose(-2/m*R[i,:,:]*Cd*transpose(R[i,:,:])*s[i]*relative_velocity[i,:]-2*constant_term[i,:]/m)*R[i,:,:]*Cd*transpose(R[i,:,:])*(relative_velocity[i,:]*transpose(relative_velocity[i,:])+s[i]^2*I3)) for i in 1:timesteps],dims=1)
sdLdw=Matrix{Expression}(undef,timesteps,3)
sdLdw.=s_times_accel_loss_deriv;
for wind_idx in 1:timesteps
    #each wind shows up in change ids wind_idx-1, wind_idx,wind_idx+1 if all present
    if wind_idx-1>0
        from_previous=differentiate(wind_change_losses[wind_idx-1],w[wind_idx,:])
        sdLdw[wind_idx,:]+=s[wind_idx]*from_previous
    end
    if wind_idx<timesteps
        from_current=differentiate(wind_change_losses[wind_idx],w[wind_idx,:])
        sdLdw[wind_idx,:]+=s[wind_idx]*from_current
    end
    if wind_idx<timesteps-1
        from_next=differentiate(wind_change_losses[wind_idx+1],w[wind_idx,:])
        sdLdw[wind_idx,:]+=s[wind_idx]*from_next
    end
end
# dLdw=differentiate(loss,w);
# dLdw_subs=subs(dLdw,airspeed=>s)
airspeed_constraint=[s[i]^2-relative_velocity[i,:]'*relative_velocity[i,:] for i in 1:timesteps]
system_state_vec=[vec(w);s]
system_param_vec=[m;g;vec(Cd);vec(a_o);vec(u);vec(R);vec(v);vec(Σ⁻)];
system=System([vec(sdLdw);airspeed_constraint],system_state_vec,system_param_vec);

In [228]:
randomized_exprs=subs([vec(sdLdw);airspeed_constraint],system_param_vec=>rand(ComplexF64,size(system_param_vec)));

In [229]:
function is_used(var,expr)
    used=variables(expr)
    if var in used
        return 1;
    else
        return 0;
    end
end
reordered_vars=vcat([[w[i,:];s[i]] for i in 1:timesteps]...);
[
reordered_vars';
stack([[is_used(vi,expr) for vi in reordered_vars] for expr in randomized_exprs[[1 5 9 13 2 6 10 14 3 7 11 15 4 8 12 16]]],dims=1)
]'

16×17 adjoint(::Matrix{Expression}) with eltype Expression:
 w₁₋₁  1  1  1  1  1  1  1  0  0  0  0  0  0  0  0  0
 w₁₋₂  1  1  1  1  1  1  1  0  0  0  0  0  0  0  0  0
 w₁₋₃  1  1  1  1  1  1  1  0  0  0  0  0  0  0  0  0
   s₁  1  1  1  1  0  0  0  0  0  0  0  0  0  0  0  0
 w₂₋₁  1  1  1  0  1  1  1  1  1  1  1  0  0  0  0  0
 w₂₋₂  1  1  1  0  1  1  1  1  1  1  1  0  0  0  0  0
 w₂₋₃  1  1  1  0  1  1  1  1  1  1  1  0  0  0  0  0
   s₂  0  0  0  0  1  1  1  1  0  0  0  0  0  0  0  0
 w₃₋₁  0  0  0  0  1  1  1  0  1  1  1  1  1  1  1  0
 w₃₋₂  0  0  0  0  1  1  1  0  1  1  1  1  1  1  1  0
 w₃₋₃  0  0  0  0  1  1  1  0  1  1  1  1  1  1  1  0
   s₃  0  0  0  0  0  0  0  0  1  1  1  1  0  0  0  0
 w₄₋₁  0  0  0  0  0  0  0  0  1  1  1  0  1  1  1  1
 w₄₋₂  0  0  0  0  0  0  0  0  1  1  1  0  1  1  1  1
 w₄₋₃  0  0  0  0  0  0  0  0  1  1  1  0  1  1  1  1
   s₄  0  0  0  0  0  0  0  0  0  0  0  0  1  1  1  1

In [157]:
randomized_accel_loss_derivs=subs([vec(s_times_accel_loss_deriv);airspeed_constraint],system_param_vec=>rand(ComplexF64,size(system_param_vec)));
[
reordered_vars';
stack([[is_used(vi,expr) for vi in reordered_vars] for expr in randomized_accel_loss_derivs[[1 5 9 13 2 6 10 14 3 7 11 15 4 8 12 16]]],dims=1)
]'

16×17 adjoint(::Matrix{Expression}) with eltype Expression:
 w₁₋₁  1  1  1  1  0  0  0  0  0  0  0  0  0  0  0  0
 w₁₋₂  1  1  1  1  0  0  0  0  0  0  0  0  0  0  0  0
 w₁₋₃  1  1  1  1  0  0  0  0  0  0  0  0  0  0  0  0
   s₁  1  1  1  1  0  0  0  0  0  0  0  0  0  0  0  0
 w₂₋₁  0  0  0  0  1  1  1  1  0  0  0  0  0  0  0  0
 w₂₋₂  0  0  0  0  1  1  1  1  0  0  0  0  0  0  0  0
 w₂₋₃  0  0  0  0  1  1  1  1  0  0  0  0  0  0  0  0
   s₂  0  0  0  0  1  1  1  1  0  0  0  0  0  0  0  0
 w₃₋₁  0  0  0  0  0  0  0  0  1  1  1  1  0  0  0  0
 w₃₋₂  0  0  0  0  0  0  0  0  1  1  1  1  0  0  0  0
 w₃₋₃  0  0  0  0  0  0  0  0  1  1  1  1  0  0  0  0
   s₃  0  0  0  0  0  0  0  0  1  1  1  1  0  0  0  0
 w₄₋₁  0  0  0  0  0  0  0  0  0  0  0  0  1  1  1  1
 w₄₋₂  0  0  0  0  0  0  0  0  0  0  0  0  1  1  1  1
 w₄₋₃  0  0  0  0  0  0  0  0  0  0  0  0  1  1  1  1
   s₄  0  0  0  0  0  0  0  0  0  0  0  0  1  1  1  1

In [226]:
wind_change_loss_derivatives=Matrix{Expression}(undef,timesteps,3)
wind_change_loss_derivatives[:].=0
for wind_idx in 1:timesteps
    #each wind shows up in change ids wind_idx-1, wind_idx,wind_idx+1 if all present
    if wind_idx-1>0
        from_previous=differentiate(wind_change_losses[wind_idx-1],w[wind_idx,:])
        wind_change_loss_derivatives[wind_idx,:]+=s[wind_idx]*from_previous
    end
    if wind_idx<timesteps
        from_current=differentiate(wind_change_losses[wind_idx],w[wind_idx,:])
        wind_change_loss_derivatives[wind_idx,:]+=s[wind_idx]*from_current
    end
    if wind_idx<timesteps-1
        from_next=differentiate(wind_change_losses[wind_idx+1],w[wind_idx,:])
        wind_change_loss_derivatives[wind_idx,:]+=s[wind_idx]*from_next
    end
end
randomized_wind_change_loss_derivs=subs([vec(wind_change_loss_derivatives);airspeed_constraint],system_param_vec=>rand(ComplexF64,size(system_param_vec)));
[
reordered_vars';
stack([[is_used(vi,expr) for vi in reordered_vars] for expr in randomized_wind_change_loss_derivs[[1 5 9 13 2 6 10 14 3 7 11 15 4 8 12 16]]],dims=1)
]'


16×17 adjoint(::Matrix{Expression}) with eltype Expression:
 w₁₋₁  1  1  1  1  1  1  1  0  0  0  0  0  0  0  0  0
 w₁₋₂  1  1  1  1  1  1  1  0  0  0  0  0  0  0  0  0
 w₁₋₃  1  1  1  1  1  1  1  0  0  0  0  0  0  0  0  0
   s₁  1  1  1  1  0  0  0  0  0  0  0  0  0  0  0  0
 w₂₋₁  1  1  1  0  1  1  1  1  1  1  1  0  0  0  0  0
 w₂₋₂  1  1  1  0  1  1  1  1  1  1  1  0  0  0  0  0
 w₂₋₃  1  1  1  0  1  1  1  1  1  1  1  0  0  0  0  0
   s₂  0  0  0  0  1  1  1  1  0  0  0  0  0  0  0  0
 w₃₋₁  0  0  0  0  1  1  1  0  1  1  1  1  1  1  1  0
 w₃₋₂  0  0  0  0  1  1  1  0  1  1  1  1  1  1  1  0
 w₃₋₃  0  0  0  0  1  1  1  0  1  1  1  1  1  1  1  0
   s₃  0  0  0  0  0  0  0  0  1  1  1  1  0  0  0  0
 w₄₋₁  0  0  0  0  0  0  0  0  1  1  1  0  1  1  1  1
 w₄₋₂  0  0  0  0  0  0  0  0  1  1  1  0  1  1  1  1
 w₄₋₃  0  0  0  0  0  0  0  0  1  1  1  0  1  1  1  1
   s₄  0  0  0  0  0  0  0  0  0  0  0  0  1  1  1  1

In [225]:
wind_change_loss_derivatives[1,:]

3-element Vector{Expression}:
 0
 0
 0

In [221]:
variables.(differentiate(wind_change_losses[1],w[2,:]))

3-element Vector{Vector{Variable}}:
 [w₁₋₁, w₂₋₁, w₁₋₂, w₂₋₂, w₁₋₃, w₂₋₃, Σ⁻₁₋₁, Σ⁻₂₋₁, Σ⁻₃₋₁, Σ⁻₁₋₂, Σ⁻₁₋₃]
 [w₁₋₁, w₂₋₁, w₁₋₂, w₂₋₂, w₁₋₃, w₂₋₃, Σ⁻₂₋₁, Σ⁻₁₋₂, Σ⁻₂₋₂, Σ⁻₃₋₂, Σ⁻₂₋₃]
 [w₁₋₁, w₂₋₁, w₁₋₂, w₂₋₂, w₁₋₃, w₂₋₃, Σ⁻₃₋₁, Σ⁻₃₋₂, Σ⁻₁₋₃, Σ⁻₂₋₃, Σ⁻₃₋₃]

In [45]:
m_val=1#pick mass units appropriately
g_val=1#pick time/length units appropriately
v_val=randn(timesteps,3)
u_val=randn(timesteps,4)
Cd_val=rand(3,3)
w_true=randn(timesteps,3)
R_val=stack([[1 0 0;0 1 0;0 0 1] for i in 1:timesteps],dims=1);
sensor_error=randn(timesteps,3)*0.1;
a_true=evaluate(a_predicted,[m;g;vec(Cd);vec(u);vec(R);vec(v);vec(w)]=>[m_val;g_val;vec(Cd_val);vec(u_val);vec(R_val);vec(v_val);vec(w_true)]);
a_val=sensor_error+a_true;
result=solve(system,target_parameters=[m_val;g_val;vec(Cd_val);vec(a_val);vec(u_val);vec(R_val);vec(v_val)])

Tracking 8464 paths... 100%|████████████████████████████| Time: 0:05:26
                   # paths tracked: 8464
   # non-singular solutions (real): 16 (4)
       # singular endpoints (real): 367 (79)
          # total solutions (real): 383 (83)


Result with 160 solutions
• 8464 paths tracked
• 16 non-singular solutions (4 real)
• 144 singular solutions (7 real)
• random_seed: 0xe73d3742
• start_system: :polyhedral
• multiplicity table of singular solutions:
╭───────┬───────┬────────┬────────────╮
│ mult. │ total │ # real │ # non-real │
├───────┼───────┼────────┼────────────┤
│   1   │  101  │   1    │    100     │
│   2   │  21   │   1    │     20     │
│   3   │   2   │   0    │     2      │
│   4   │   4   │   0    │     4      │
│   8   │   8   │   4    │     4      │
│   9   │   2   │   0    │     2      │
│  14   │   1   │   0    │     1      │
│  15   │   2   │   0    │     2      │
│  16   │   2   │   0    │     2      │
│  44   │   1   │   1    │     0      │
╰───────┴───────┴────────┴────────────╯


In [49]:
physical_sols=results(result,only_real=true)
losses=stack([evaluate(loss,[m;g;vec(Cd);vec(u);vec(R);vec(v);vec(w);vec(a_o)]=>[m_val;g_val;vec(Cd_val);vec(u_val);vec(R_val);vec(v_val);real.(wi.solution[1:3*timesteps]);vec(a_val)])[:] for wi in physical_sols],dims=1)

11×2 Matrix{Float64}:
 9.03633      5.09755e-30
 2.25908      1.71801e-26
 9.03633      1.35049
 2.25908      1.35049
 2.92387e-29  5.09755e-30
 2.62149e-28  1.35049
 2.92387e-29  5.40197
 2.25908      1.35049
 2.25908      1.35049
 2.25908      5.40197
 9.03633      5.40197

In [55]:
#best estimates
optimal_candidates=stack([[sol.return_code;sol.solution] for (i,sol) in enumerate(physical_sols) if all(losses[i,:].<1e-10)],dims=2)

9×1 Matrix{Any}:
                       :success
  0.43057680376262525 + 6.62790628654034e-63im
   1.0137182241094582 + 3.0538424910233095e-52im
   0.9355726187322264 - 3.9881384532191155e-62im
 -0.05320006782523078 - 6.796757133867263e-51im
  -1.1812139113158702 + 2.9854064992668807e-62im
  -1.4928747141868441 + 4.3014807053046274e-51im
   1.8006577079533592 - 2.63596960622292e-62im
    1.250905571828404 - 2.0942964690041264e-51im

In [56]:
#true value
w_true

2×3 Matrix{Float64}:
 0.283629  -0.560944   0.0786644
 0.403271  -0.91228   -0.812657

In [57]:
#loss at true value
evaluate(loss,[m;g;vec(Cd);vec(u);vec(R);vec(v);vec(w);vec(a_o)]=>[m_val;g_val;vec(Cd_val);vec(u_val);vec(R_val);vec(v_val);vec(w_true);vec(a_val)])

2×1 Matrix{Float64}:
 0.06226448700369933
 0.05230938266040326

In [58]:
#random sensor error
[a_true sensor_error a_val]

2×9 Matrix{Float64}:
  0.316659   0.194979  -1.21585  …   0.38179   -0.0424444  -1.1752
 -0.298097  -0.379808  -2.65493     -0.114567  -0.357171   -2.52035

In [ ]:
#generic parameter values
function hat(v)
    A=[0    -v[3]  v[2];
        v[3]  0    -v[1];
        -v[2]  v[1]  0];
    return A
end

function random_rotmat(complex=false)
    if complex
        temp=rand(ComplexF64,3)
        random_magnitude=pi*rand(ComplexF64)
    else
        temp=rand(Float64,3)
        random_magnitude=pi*rand(Float64)
    end
    temp_norm=temp'*temp
    random_axis=temp/temp_norm
    random_rotvec=random_magnitude*random_axis
    return exp(hat(random_rotvec));
end
generic_m_val=1#pick mass units appropriately
generic_g_val=1#pick time/length units appropriately
generic_v_val=randn(ComplexF64,3)
generic_u_val=randn(ComplexF64,4)
generic_w_true=randn(ComplexF64,3)
generic_Cd_val=rand(ComplexF64,3,3)
generic_R_val=random_rotmat(true);
generic_sensor_error=zeros(3)#randn(3)*0.1;
generic_a_true=evaluate(a_predicted,[m;g;vec(Cd);u;vec(R);v;w]=>[generic_m_val;generic_g_val;vec(generic_Cd_val);generic_u_val;vec(generic_R_val);generic_v_val;generic_w_true]);
generic_a_val=generic_sensor_error+generic_a_true;
generic_result=solve(system,target_parameters=[generic_m_val;generic_g_val;vec(generic_Cd_val);generic_a_val;generic_u_val;vec(generic_R_val);generic_v_val])

In [ ]:
transpose([generic_w_true [s.solution[1:3] for s in results(generic_result,only_nonsingular=true)]...])

In [ ]:
vars=[m;g;vec(Cd);u;vec(R);v;w;a_o]
generic_solution_values=[[generic_m_val;generic_g_val;vec(generic_Cd_val);generic_u_val;vec(generic_R_val);generic_v_val;wi.solution[1:3];generic_a_val] for wi in results(generic_result,only_nonsingular=true)]
losses=[evaluate(loss,vars=>vals) for vals in generic_solution_values]

In [ ]:
acceleration_errors=[evaluate(error,vars=>vals) for vals in generic_solution_values]

In [ ]:
optimal_accelerations=[evaluate(a_predicted,vars=>vals) for vals in generic_solution_values]

In [ ]:
generic_a_val